# SkateFormer  Fine-tuning głowicy (`fc`) na klasach interakcyjnych, wynik negatywny

Strategia: zamrażam cały ekstraktor cech i douczam tylko ostatnią warstwę (`fc`)  
Dane treningowe: `NTU60_CS.npz` wagi strat 3x wyższe dla 8 klas interakcyjnych  
Ewaluacja: na `PKU_XSub_both.npz` i `PKU_XView_both.npz` po każdej epoce, plus kontrolnie `NTU60_CS.npz` / `NTU60_CV.npz`  
Klasy interakcyjne (NTU idx): 49, 50, 51, 52, 53, 54, 55, 57  
Wynik ekspertyment nie przyniósł poprawy na PKU-MMD, a na NTU dokładność lekko spadła


In [ ]:
!git clone https://github.com/KAIST-VICLab/SkateFormer.git
import os
os.chdir('/content/SkateFormer')
!ls

Cloning into 'SkateFormer'...
remote: Enumerating objects: 180, done.
remote: Counting objects: 100% (79/79), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 180 (delta 52), reused 43 (delta 35), pack-reused 101 (from 1)
Receiving objects: 100% (180/180), 1.12 MiB | 6.02 MiB/s, done.
Resolving deltas: 100% (80/80), done.
assets	data	 LICENSE  model		  README.md	     skateformer
config	feeders  main.py  pyproject.toml  requirements.yaml  torchlight


In [ ]:
!pip install -q einops==0.6.1 timm==0.9.12 tensorpack torchpack loguru msgpack msgpack-numpy tabulate tensorboardX

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.3/296.3 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 11.0 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil, numpy as np, os

DRIVE_ROOT    = '/content/drive/MyDrive'
DRIVE_WEIGHTS = f'{DRIVE_ROOT}/SkateFormer_weights/ntu60_CSub'

os.makedirs('/content/SkateFormer/data/ntu', exist_ok=True)

for fname in ['NTU60_CS.npz', 'NTU60_CV.npz', 'PKU_XSub_both.npz', 'PKU_XView_both.npz']:
    src = f'{DRIVE_ROOT}/PKU_data/{fname}' if 'PKU' in fname else f'{DRIVE_ROOT}/{fname}'
    dst = f'/content/SkateFormer/data/ntu/{fname}'
    if not os.path.exists(dst):
        shutil.copy(src, dst)
        d = np.load(dst)
        print(f'Skopiowano: {fname}  shape={d["x_train"].shape if "x_train" in d else d["x_test"].shape}')
    else:
        print(f'Już istnieje: {fname}')

# Wagi
pt_src = f'{DRIVE_WEIGHTS}/SkateFormer_j.pt'
pt_dst = '/content/SkateFormer/SkateFormer_j.pt'
if not os.path.exists(pt_dst):
    shutil.copy(pt_src, pt_dst)
    print(f'Skopiowano: SkateFormer_j.pt  ({os.path.getsize(pt_dst)/1024**2:.1f} MB)')
else:
    print('Wagi już istnieją.')


Skopiowano: NTU60_CS.npz  shape=(40091, 300, 150)
Skopiowano: NTU60_CV.npz  shape=(37646, 300, 150)
Skopiowano: PKU_XSub_both.npz  shape=(18713, 300, 150)
Skopiowano: PKU_XView_both.npz  shape=(14268, 300, 150)
Skopiowano: SkateFormer_j.pt  (13.9 MB)


In [ ]:
import os, sys
os.chdir('/content/SkateFormer')

#Poprawka torchlight
with open('/content/SkateFormer/torchlight/torchlight/__init__.py', 'w') as f:
    f.write('''import argparse

class DictAction(argparse.Action):
    def __call__(self, parser, namespace, values, option_string=None):
        input_dict = getattr(namespace, self.dest, {}) or {}
        for kv in values:
            key, val = kv.split("=", 1)
            try:
                val = eval(val)
            except:
                pass
            input_dict[key] = val
        setattr(namespace, self.dest, input_dict)

class IO:
    pass
''')
print('Zastosowano poprawkę torchlight.')

#Usunięcie konfliktu z zainstalowaną wersją torchlight
os.system('pip uninstall torchlight -y')
if 'torchlight' in sys.modules:
    del sys.modules['torchlight']
sys.path.insert(0, '/content/SkateFormer/torchlight')
print('Usunięto konflikt z systemowym torchlight.')

#Poprawka main.py
with open('/content/SkateFormer/main.py', 'r') as f:
    content = f.read()
content = content.replace(
    'from torchlight import DictAction',
    '''class DictAction(argparse.Action):
    def __call__(self, parser, namespace, values, option_string=None):
        input_dict = getattr(namespace, self.dest, {}) or {}
        for kv in values:
            key, val = kv.split("=", 1)
            try:
                val = eval(val)
            except:
                pass
            input_dict[key] = val
        setattr(namespace, self.dest, input_dict)'''
)
content = content.replace('default_arg = yaml.load(f)',
                          'default_arg = yaml.load(f, Loader=yaml.SafeLoader)')
with open('/content/SkateFormer/main.py', 'w') as f:
    f.write(content)
print('Zastosowano poprawkę main.py.')

#Zastąpienie przestarzałego np.int
with open('/content/SkateFormer/feeders/feeder_ntu.py', 'r') as f:
    content = f.read()
content = content.replace('.astype(np.int)', '.astype(int)')
with open('/content/SkateFormer/feeders/feeder_ntu.py', 'w') as f:
    f.write(content)
print('Zastąpiono przestarzały typ np.int.')

#Zastąpienie przestarzałych typów NumPy
with open('/content/SkateFormer/feeders/tools.py', 'r') as f:
    content = f.read()
for old, new in [('np.int)', 'int)'), ('np.int,', 'int,'),
                 ('np.float)', 'float)'), ('np.float,', 'float,'),
                 ('np.bool)', 'bool)'), ('np.complex)', 'complex)')]:
    content = content.replace(old, new)
with open('/content/SkateFormer/feeders/tools.py', 'w') as f:
    f.write(content)
print('Zastąpiono przestarzałe typy NumPy.')

print('\nZakończono dostosowanie kodu SkateFormer.')

Zastosowano poprawkę torchlight.
Usunięto konflikt z systemowym torchlight.
Zastosowano poprawkę main.py.
Zastąpiono przestarzały typ np.int.
Zastąpiono przestarzałe typy NumPy.

Zakończono dostosowanie kodu SkateFormer.


In [ ]:
#Fine-tuning głowicy
import torch
import torch.nn as nn
import numpy as np
import sys, os

sys.path.insert(0, '/content/SkateFormer')
sys.path.insert(0, '/content/SkateFormer/torchlight')

from feeders.feeder_ntu import Feeder
from model.SkateFormer import SkateFormer_

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

model = SkateFormer_(
    num_classes=60, num_people=2, num_points=24, kernel_size=7,
    num_heads=32, attn_drop=0.5, head_drop=0.0, rel=True,
    drop_path=0.2, type_1_size=[8,8], type_2_size=[8,12],
    type_3_size=[8,8], type_4_size=[8,12], mlp_ratio=4.0, index_t=True
)

weights = torch.load('/content/SkateFormer/SkateFormer_j.pt', map_location=DEVICE)
if 'model' in weights:
    weights = weights['model']
model.load_state_dict(weights, strict=False)
model = model.to(DEVICE)

for name, param in model.named_parameters():
    param.requires_grad = False

fc_layer = None
for name, module in model.named_modules():
    if isinstance(module, nn.Linear):
        fc_name = name
        fc_layer = module

print(f'Ostatnia warstwa liniowa: {fc_name}  ({fc_layer.in_features} → {fc_layer.out_features})')

for param in fc_layer.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Parametry treningowe: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)')

INTERACTION_NTU = [49, 50, 51, 52, 53, 54, 55, 57]
INTERACTION_WEIGHT = 3.0
class_weights = torch.ones(60, device=DEVICE)
for idx in INTERACTION_NTU:
    class_weights[idx] = INTERACTION_WEIGHT
print(f'Waga dla klas interakcyjnych: {INTERACTION_WEIGHT}x')

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(fc_layer.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)

train_feeder = Feeder(
    data_path='./data/ntu/NTU60_CS.npz',
    split='train', debug=False,
    window_size=64, p_interval=[0.5, 1], aug_method='a123489',
    intra_p=0.5, inter_p=0.2, thres=64, uniform=True, partition=True
)
train_loader = torch.utils.data.DataLoader(
    train_feeder, batch_size=64, shuffle=True, num_workers=4, pin_memory=True
)
print(f'Train samples: {len(train_feeder)}')

NUM_EPOCHS = 5
os.makedirs('./work_dir/finetune', exist_ok=True)

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    n = 0

    for batch_idx, (data, index_t, label, _) in enumerate(train_loader):
        data    = data.float().to(DEVICE)
        index_t = index_t.to(DEVICE)
        label   = label.long().to(DEVICE)

        optimizer.zero_grad()
        output = model(data, index_t=index_t)
        loss   = criterion(output, label)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(label)
        correct    += (output.argmax(1) == label).sum().item()
        n          += len(label)

        if batch_idx % 100 == 0:
            print(f'Epoka {epoch+1}/{NUM_EPOCHS}  batch {batch_idx}/{len(train_loader)}  '
                  f'loss={total_loss/n:.4f}  acc={correct/n*100:.2f}%', end='\r')

    scheduler.step()
    print(f'\nEpoka {epoch+1} — loss={total_loss/n:.4f}  train_acc={correct/n*100:.2f}%')

    torch.save(model.state_dict(), f'./work_dir/finetune/epoch{epoch+1}.pt')
    print(f'  Zapisano: epoch{epoch+1}.pt')

print('\nFine-tuning zakończony!')


Device: cuda
Ostatnia warstwa liniowa: head  (192 → 60)
Parametry treningowe: 11,580 / 3,616,083 (0.32%)
Waga dla klas interakcyjnych: 3.0x
Train samples: 40091

Epoka 1 — loss=0.1010  train_acc=98.87%
  Zapisano: epoch1.pt

Epoka 2 — loss=0.0695  train_acc=98.83%
  Zapisano: epoch2.pt

Epoka 3 — loss=0.0599  train_acc=98.82%
  Zapisano: epoch3.pt

Epoka 4 — loss=0.0540  train_acc=98.84%
  Zapisano: epoch4.pt

Epoka 5 — loss=0.0565  train_acc=98.73%
  Zapisano: epoch5.pt

Fine-tuning zakończony!


In [ ]:
import torch, numpy as np, pickle
from feeders.feeder_ntu import Feeder
from model.SkateFormer import SkateFormer_

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

INTERACTION_NTU = [49, 50, 51, 52, 53, 54, 55, 57]
INT_NAMES = {
    49: 'punch/slap', 50: 'kick person', 51: 'push person', 52: 'pat on back',
    53: 'point at person', 54: 'hugging', 55: 'giving object', 57: 'handshaking'
}

EVAL_SETS = [
    ('PKU XSub', './data/ntu/PKU_XSub_both.npz'),
    ('PKU XView', './data/ntu/PKU_XView_both.npz'),
]

def evaluate(model, npz_path):
    feeder = Feeder(
        data_path=npz_path, split='test', debug=False,
        window_size=64, p_interval=[0.95], thres=64,
        uniform=True, partition=True
    )
    loader = torch.utils.data.DataLoader(
        feeder, batch_size=128, shuffle=False, num_workers=4
    )
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for data, index_t, label, _ in loader:
            data    = data.float().to(DEVICE)
            index_t = index_t.to(DEVICE)
            out     = model(data, index_t=index_t)
            all_preds.append(out.argmax(1).cpu().numpy())
            all_labels.append(label.numpy())
    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_labels)
    overall = (y_pred == y_true).mean() * 100
    mask_int = np.isin(y_true, INTERACTION_NTU)
    int_acc  = (y_pred[mask_int] == y_true[mask_int]).mean() * 100
    return y_pred, y_true, overall, int_acc

def load_model(weights_path):
    m = SkateFormer_(
        num_classes=60, num_people=2, num_points=24, kernel_size=7,
        num_heads=32, attn_drop=0.5, head_drop=0.0, rel=True,
        drop_path=0.2, type_1_size=[8,8], type_2_size=[8,12],
        type_3_size=[8,8], type_4_size=[8,12], mlp_ratio=4.0, index_t=True
    ).to(DEVICE)
    w = torch.load(weights_path, map_location=DEVICE)
    if 'model' in w: w = w['model']
    m.load_state_dict(w, strict=False)
    return m

print('Ładuję oryginalne wagi...')
model_orig = load_model('/content/SkateFormer/SkateFormer_j.pt')

results = {}
for name, npz in EVAL_SETS:
    _, _, ov, ia = evaluate(model_orig, npz)
    results[f'orig_{name}'] = (ov, ia)
    print(f'Oryginalne  {name}: Overall={ov:.2f}%  Interakcje={ia:.2f}%')

print()
for epoch in range(1, 6):
    pt_path = f'./work_dir/finetune/epoch{epoch}.pt'
    if not os.path.exists(pt_path):
        continue
    model_ft = load_model(pt_path)
    for name, npz in EVAL_SETS:
        _, _, ov, ia = evaluate(model_ft, npz)
        results[f'ep{epoch}_{name}'] = (ov, ia)
        print(f'Epoka {epoch}  {name}: Overall={ov:.2f}%  Interakcje={ia:.2f}%')

print(f'\n{"Epoka":<10} {"XSub Overall":>14} {"XSub Interakcje":>16} {"XView Overall":>14} {"XView Interakcje":>16}')
print('-' * 74)
for label, key in [('Oryginalne', 'orig'), *[(f'Epoka {e}', f'ep{e}') for e in range(1,6)]]:
    xs = results.get(f'{key}_PKU XSub')
    xv = results.get(f'{key}_PKU XView')
    xs_ov  = f'{xs[0]:.2f}%' if xs else 'N/A'
    xs_int = f'{xs[1]:.2f}%' if xs else 'N/A'
    xv_ov  = f'{xv[0]:.2f}%' if xv else 'N/A'
    xv_int = f'{xv[1]:.2f}%' if xv else 'N/A'
    print(f'{label:<10} {xs_ov:>14} {xs_int:>16} {xv_ov:>14} {xv_int:>16}')


Ładuję oryginalne wagi...
Oryginalne  PKU XSub: Overall=62.61%  Interakcje=84.10%
Oryginalne  PKU XView: Overall=62.79%  Interakcje=89.84%

Epoka 1  PKU XSub: Overall=62.54%  Interakcje=84.62%
Epoka 1  PKU XView: Overall=62.93%  Interakcje=90.02%
Epoka 2  PKU XSub: Overall=62.57%  Interakcje=84.10%
Epoka 2  PKU XView: Overall=62.72%  Interakcje=90.02%
Epoka 3  PKU XSub: Overall=62.54%  Interakcje=84.62%
Epoka 3  PKU XView: Overall=62.82%  Interakcje=89.84%
Epoka 4  PKU XSub: Overall=62.65%  Interakcje=84.62%
Epoka 4  PKU XView: Overall=62.86%  Interakcje=90.37%
Epoka 5  PKU XSub: Overall=62.61%  Interakcje=84.62%
Epoka 5  PKU XView: Overall=63.04%  Interakcje=90.37%

Epoka        XSub Overall  XSub Interakcje  XView Overall XView Interakcje
--------------------------------------------------------------------------
Oryginalne         62.61%           84.10%         62.79%           89.84%
Epoka 1            62.54%           84.62%         62.93%           90.02%
Epoka 2            62.57

In [ ]:
EVAL_NTU = [
    ('NTU CS', './data/ntu/NTU60_CS.npz'),
    ('NTU CV', './data/ntu/NTU60_CV.npz'),
]

model_orig = load_model('/content/SkateFormer/SkateFormer_j.pt')
for name, npz in EVAL_NTU:
    _, _, ov, ia = evaluate(model_orig, npz)
    results[f'orig_{name}'] = (ov, ia)
    print(f'Oryginalne  {name}: Overall={ov:.2f}%  Interakcje={ia:.2f}%')

print()

for epoch in range(1, 6):
    pt_path = f'./work_dir/finetune/epoch{epoch}.pt'
    if not os.path.exists(pt_path):
        continue
    model_ft = load_model(pt_path)
    for name, npz in EVAL_NTU:
        _, _, ov, ia = evaluate(model_ft, npz)
        results[f'ep{epoch}_{name}'] = (ov, ia)
        print(f'Epoka {epoch}  {name}: Overall={ov:.2f}%  Interakcje={ia:.2f}%')

print(f'\n{"Epoka":<10} {"NTU CS Overall":>16} {"NTU CS Interakcje":>18} {"NTU CV Overall":>16} {"NTU CV Interakcje":>18}')
print('-' * 82)
for label, key in [('Oryginalne', 'orig'), *[(f'Epoka {e}', f'ep{e}') for e in range(1,6)]]:
    cs = results.get(f'{key}_NTU CS')
    cv = results.get(f'{key}_NTU CV')
    print(f'{label:<10} {cs[0]:>15.2f}% {cs[1]:>17.2f}% {cv[0]:>15.2f}% {cv[1]:>17.2f}%' if cs and cv else f'{label:<10}  N/A')

Oryginalne  NTU CS: Overall=92.62%  Interakcje=96.42%
Oryginalne  NTU CV: Overall=98.26%  Interakcje=99.40%

Epoka 1  NTU CS: Overall=92.53%  Interakcje=96.42%
Epoka 1  NTU CV: Overall=98.23%  Interakcje=99.40%
Epoka 2  NTU CS: Overall=92.58%  Interakcje=96.42%
Epoka 2  NTU CV: Overall=98.26%  Interakcje=99.40%
Epoka 3  NTU CS: Overall=92.54%  Interakcje=96.46%
Epoka 3  NTU CV: Overall=98.24%  Interakcje=99.40%
Epoka 4  NTU CS: Overall=92.50%  Interakcje=96.46%
Epoka 4  NTU CV: Overall=98.24%  Interakcje=99.40%
Epoka 5  NTU CS: Overall=92.56%  Interakcje=96.42%
Epoka 5  NTU CV: Overall=98.23%  Interakcje=99.40%

Epoka        NTU CS Overall  NTU CS Interakcje   NTU CV Overall  NTU CV Interakcje
----------------------------------------------------------------------------------
Oryginalne           92.62%             96.42%           98.26%             99.40%
Epoka 1              92.53%             96.42%           98.23%             99.40%
Epoka 2              92.58%             96.42%  

Wniosek: fine-tuning wyłącznie ostatniej warstwy liniowej (0,32% parametrów) z 3x wagą dla klas interakcyjnych nie daje istotnej poprawy na PKU-MMD (zmiany rzędu ±0,4pp, w granicach szumu) i nieznacznie obniża dokładność na natywnym NTU (spadek ~0,1pp na CS). Ogranicznik transferu leży w ekstraktorze cech, nie w warstwie klasyfikującej.